# Local Debug & Model Improvement Notebook

**Goal:** Diagnose why AUC was stuck at ~0.68, find what's hurting performance, and iterate on features/models locally.

**Key findings so far:**
- Original pipeline AUC: ~0.677 (LGBM ordinal)
- After fixes: **0.694** (LGBM+XGB ensemble with target+WOE encoding)

**Root causes of poor performance:**
1. LGBM was early-stopping on `logloss` instead of `auc` -> barely trained (4 iterations!)
2. v3 target+WOE pipeline was broken (WOE mapped on already-encoded columns -> all zeros)
3. Missing-value indicators were not being used as features
4. Several useful ratio/flag features were missing

Run this notebook on your local machine (CPU). No GPU needed.

In [2]:
# -- Move to project root & install all required packages --
import os, sys

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if os.path.basename(os.getcwd()) == 'notebooks':
    os.chdir(project_root)
print(f'Working directory: {os.getcwd()}')

!{sys.executable} -m pip install -q lightgbm xgboost shap scipy scikit-learn pandas numpy matplotlib seaborn category_encoders pydantic pyyaml pyarrow optuna
!{sys.executable} -m pip install -q -e .

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.metrics import roc_auc_score, classification_report
from lightgbm import LGBMClassifier, early_stopping
import xgboost as xgb
from scipy.stats import rankdata

sns.set_style('whitegrid')
%matplotlib inline

from src.utils.seeding import seed_everything
from src.data.loader import load_train, load_test, load_sample_submission, TARGET_COL, ID_COL
from src.data.splits import make_folds

seed_everything(42)

SEED = 42
N_FOLDS = 5

print(f'Python: {sys.version}')

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
ERROR: file:///C:/Users/Tinevimbo/Desktop/deep_learning_indabaX/notebooks does not appear to be a Python project: neither 'setup.py' nor 'pyproject.toml' found.


ModuleNotFoundError: No module named 'src'

## 1. Load Data

In [ ]:
train_raw = load_train()
test_raw = load_test()
sample_sub = load_sample_submission()
folds_df = make_folds(train_raw, n_folds=N_FOLDS, seed=SEED)

print(f'Train: {train_raw.shape}')
print(f'Test:  {test_raw.shape}')
print(f'Target balance:\n{train_raw[TARGET_COL].value_counts(normalize=True)}')

## 2. Deep EDA — What Separates Defaults?

In [ ]:
y_raw = train_raw[TARGET_COL].values
num_cols = ['amount_usd', 'annual_rate_pct', 'term_months',
            'num_dependents', 'months_at_employer',
            'monthly_income_usd', 'existing_obligations']

print('=== Univariate AUC per raw numeric feature ===')
for col in num_cols:
    vals = train_raw[col].fillna(train_raw[col].median())
    try:
        auc = roc_auc_score(y_raw, vals)
        auc = max(auc, 1 - auc)
        print(f'  {col:30s}  AUC = {auc:.4f}')
    except Exception:
        print(f'  {col:30s}  SKIPPED')

In [ ]:
cat_cols = ['product_code', 'payment_frequency', 'loan_purpose',
            'client_gender', 'marital_status', 'employment_sector',
            'collateral_type', 'disbursement_channel', 'province']

fig, axes = plt.subplots(3, 3, figsize=(18, 14))
for ax, col in zip(axes.ravel(), cat_cols):
    rates = train_raw.groupby(col)[TARGET_COL].mean().sort_values(ascending=False)
    rates.plot.barh(ax=ax, color='steelblue')
    ax.set_title(f'Default rate by {col}')
    ax.set_xlabel('Default rate')
plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for label, color in [(0, 'green'), (1, 'red')]:
    subset = train_raw[train_raw[TARGET_COL] == label]['annual_rate_pct'].dropna()
    axes[0].hist(subset, bins=50, alpha=0.5, label=f'Target={label}', color=color, density=True)
axes[0].set_title('annual_rate_pct distribution by Target')
axes[0].legend()

for label, color in [(0, 'green'), (1, 'red')]:
    subset = train_raw[train_raw[TARGET_COL] == label]['amount_usd'].dropna()
    axes[1].hist(np.log1p(subset), bins=50, alpha=0.5, label=f'Target={label}', color=color, density=True)
axes[1].set_title('log(amount_usd) distribution by Target')
axes[1].legend()

plt.tight_layout()
plt.show()

## 3. Feature Engineering — Two Variants

In [ ]:
from src.features.base import BaseFeatureEngineer
from src.features.encoders.ordinal import OrdinalEncoder
from src.features.encoders.target import KFoldTargetEncoder
from src.features.encoders.woe import WOEEncoder
from src.features.encoders.group_stats import GroupStatsEncoder

fe = BaseFeatureEngineer()
y = train_raw[TARGET_COL].copy()
train_fe = fe.fit_transform(train_raw)
test_fe = fe.transform(test_raw)

print(f'Columns after base FE: {train_fe.shape[1]}')
print(f'New features: {set(train_fe.columns) - set(train_raw.columns)}')

In [ ]:
def build_v2_ordinal(train_fe, test_fe):
    """Ordinal encoding for tree models."""
    enc = OrdinalEncoder()
    enc.fit(train_fe)
    return enc.transform(train_fe), enc.transform(test_fe)


def build_v3_target_woe(train_fe, test_fe, y, folds_df):
    """Target + WOE + group stats encoding."""
    cat_cols = [c for c in train_fe.columns if train_fe[c].dtype.name in ('category', 'object')]

    woe = WOEEncoder(cat_cols=cat_cols)
    woe.fit(train_fe, y)
    tr_woe = woe.transform(train_fe[cat_cols].copy()).rename(columns={c: f'{c}_woe' for c in cat_cols})
    te_woe = woe.transform(test_fe[cat_cols].copy()).rename(columns={c: f'{c}_woe' for c in cat_cols})

    gs = GroupStatsEncoder()
    gs.fit(train_fe, y)
    tr_gs = gs.transform(train_fe[['province', 'employment_sector']].copy())
    te_gs = gs.transform(test_fe[['province', 'employment_sector']].copy())
    gs_new = [c for c in tr_gs.columns if c not in ('province', 'employment_sector')]

    te_enc = KFoldTargetEncoder(smoothing=10.0)
    te_enc.fit(train_fe, y, folds_df)
    tr_enc = te_enc.transform_train(train_fe, y, folds_df)
    te_enc_out = te_enc.transform(test_fe)

    for col in tr_woe.columns:
        tr_enc[col] = tr_woe[col].values
        te_enc_out[col] = te_woe[col].values
    for col in gs_new:
        tr_enc[col] = tr_gs[col].values
        te_enc_out[col] = te_gs[col].values

    return tr_enc, te_enc_out


def get_Xy(df, exclude_cols=None):
    """Extract X matrix and feature column names."""
    exclude = {ID_COL, TARGET_COL} | (exclude_cols or set())
    feat_cols = [c for c in df.columns
                 if c not in exclude and df[c].dtype.kind in ('i', 'f', 'u')]
    X = df[feat_cols].values.astype(np.float32)
    X = np.nan_to_num(X, nan=0.0)
    return X, feat_cols


# Build both variants
tr_v2, te_v2 = build_v2_ordinal(train_fe, test_fe)
tr_v3, te_v3 = build_v3_target_woe(train_fe, test_fe, y, folds_df)

X_v2, feat_v2 = get_Xy(tr_v2)
X_v3, feat_v3 = get_Xy(tr_v3)
Xt_v2, _ = get_Xy(te_v2)
Xt_v3, _ = get_Xy(te_v3)
yv = y.values

print(f'V2 (ordinal):    {X_v2.shape[1]} features')
print(f'V3 (target+WOE): {X_v3.shape[1]} features')

## 4. Cross-Validation — Compare Variants

In [ ]:
def cv_lgbm(X, y, label, n_splits=5, **extra_params):
    """5-fold LGBM CV with AUC metric and early stopping."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    params = dict(
        n_estimators=3000, num_leaves=31, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=SEED, verbose=-1, metric='auc',
    )
    params.update(extra_params)
    oof = np.zeros(len(X))
    for tr_idx, val_idx in skf.split(X, y):
        m = LGBMClassifier(**params)
        m.fit(X[tr_idx], y[tr_idx], eval_set=[(X[val_idx], y[val_idx])],
              callbacks=[early_stopping(200, verbose=False)])
        oof[val_idx] = m.predict_proba(X[val_idx])[:, 1]
    auc = roc_auc_score(y, oof)
    print(f'[{label:30s}] OOF AUC = {auc:.4f}')
    return oof


def cv_xgb(X, y, label, n_splits=5, **extra_params):
    """5-fold XGB CV with AUC metric and early stopping."""
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=SEED)
    neg, pos = (y == 0).sum(), (y == 1).sum()
    params = dict(
        n_estimators=3000, max_depth=6, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=float(neg / pos),
        tree_method='hist', eval_metric='auc',
        random_state=SEED, verbosity=0, early_stopping_rounds=200,
    )
    params.update(extra_params)
    oof = np.zeros(len(X))
    for tr_idx, val_idx in skf.split(X, y):
        m = xgb.XGBClassifier(**params)
        m.fit(X[tr_idx], y[tr_idx], eval_set=[(X[val_idx], y[val_idx])], verbose=False)
        oof[val_idx] = m.predict_proba(X[val_idx])[:, 1]
    auc = roc_auc_score(y, oof)
    print(f'[{label:30s}] OOF AUC = {auc:.4f}')
    return oof


print('=== LGBM on both variants ===')
oof_lgbm_v2 = cv_lgbm(X_v2, yv, 'LGBM v2_ordinal')
oof_lgbm_v3 = cv_lgbm(X_v3, yv, 'LGBM v3_target_woe')
print()

print('=== XGB on both variants ===')
oof_xgb_v2 = cv_xgb(X_v2, yv, 'XGB v2_ordinal')
oof_xgb_v3 = cv_xgb(X_v3, yv, 'XGB v3_target_woe')

## 5. Feature Importance Analysis

In [ ]:
# Train a final model on all data for feature importance
m_imp = LGBMClassifier(
    n_estimators=500, num_leaves=31, learning_rate=0.01,
    subsample=0.8, colsample_bytree=0.8,
    is_unbalance=True, random_state=SEED, verbose=-1, metric='auc',
)
m_imp.fit(X_v3, yv)

imp_df = pd.DataFrame({
    'feature': feat_v3,
    'importance': m_imp.feature_importances_,
}).sort_values('importance', ascending=True)

fig, ax = plt.subplots(figsize=(10, max(8, len(feat_v3) * 0.22)))
ax.barh(imp_df['feature'], imp_df['importance'], color='steelblue')
ax.set_title('LGBM Feature Importance (v3 variant)')
ax.set_xlabel('Split count')
plt.tight_layout()
plt.show()

print('\nTop 20 features:')
print(imp_df.tail(20).to_string(index=False))

## 6. Adversarial Validation — Train vs Test Shift

In [ ]:
adv_X = np.vstack([X_v3, Xt_v3])
adv_y = np.concatenate([np.zeros(len(X_v3)), np.ones(len(Xt_v3))])

adv_model = LGBMClassifier(n_estimators=100, num_leaves=31, verbose=-1, random_state=SEED)
adv_scores = cross_val_score(adv_model, adv_X, adv_y, cv=5, scoring='roc_auc')
print(f'Adversarial AUC: {np.mean(adv_scores):.4f} +/- {np.std(adv_scores):.4f}')
if np.mean(adv_scores) > 0.6:
    print('WARNING: Notable train/test shift!')
else:
    print('OK: Train and test distributions look similar.')

## 7. Ensembles — Combine Multiple Models

In [ ]:
print('=== Ensemble experiments ===')

# Rank average of all 4 OOF predictions
all_oofs = {
    'lgbm_v2': oof_lgbm_v2,
    'lgbm_v3': oof_lgbm_v3,
    'xgb_v2': oof_xgb_v2,
    'xgb_v3': oof_xgb_v3,
}

# Pairwise ensembles
import itertools
for (n1, o1), (n2, o2) in itertools.combinations(all_oofs.items(), 2):
    ens = (rankdata(o1) + rankdata(o2)) / 2
    print(f'  {n1:12s} + {n2:12s}  AUC = {roc_auc_score(yv, ens):.4f}')

# Full ensemble (all 4)
full_ens = sum(rankdata(o) for o in all_oofs.values()) / len(all_oofs)
print(f'\n  ALL 4 rank avg           AUC = {roc_auc_score(yv, full_ens):.4f}')

# Best v3 models only
v3_ens = (rankdata(oof_lgbm_v3) + rankdata(oof_xgb_v3)) / 2
print(f'  LGBM_v3 + XGB_v3 only    AUC = {roc_auc_score(yv, v3_ens):.4f}')

## 8. Error Analysis

In [ ]:
best_oof = oof_xgb_v3  # use best single model for analysis

# Calibration curve
from sklearn.calibration import calibration_curve

prob_true, prob_pred = calibration_curve(yv, best_oof, n_bins=15, strategy='quantile')

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

axes[0].plot(prob_pred, prob_true, 'o-', label='Model OOF')
axes[0].plot([0, 1], [0, 1], 'k--', label='Perfect')
axes[0].set_xlabel('Mean predicted probability')
axes[0].set_ylabel('Fraction of positives')
axes[0].set_title('Calibration Curve')
axes[0].legend()

# Probability distributions
axes[1].hist(best_oof[yv == 0], bins=50, alpha=0.6, label='No Default', density=True, color='green')
axes[1].hist(best_oof[yv == 1], bins=50, alpha=0.6, label='Default', density=True, color='red')
axes[1].set_title('OOF Probability Distribution')
axes[1].legend()

# ROC curve
from sklearn.metrics import roc_curve
fpr, tpr, _ = roc_curve(yv, best_oof)
axes[2].plot(fpr, tpr, color='steelblue', lw=2)
axes[2].plot([0, 1], [0, 1], 'k--')
axes[2].set_xlabel('FPR')
axes[2].set_ylabel('TPR')
axes[2].set_title(f'ROC Curve (AUC = {roc_auc_score(yv, best_oof):.4f})')

plt.tight_layout()
plt.show()

In [ ]:
# Segment-level AUC
print('=== AUC by province ===')
for prov in sorted(train_raw['province'].dropna().unique()):
    mask = (train_raw['province'] == prov).values
    if yv[mask].sum() < 5 or (1 - yv[mask]).sum() < 5:
        continue
    auc = roc_auc_score(yv[mask], best_oof[mask])
    print(f'  {prov:25s}  n={mask.sum():5d}  AUC={auc:.4f}')

print()
print('=== AUC by rate bucket ===')
rate = train_raw['annual_rate_pct'].fillna(0)
for label, mask in [('Bank (<40%)', (rate < 40).values), ('MFI (>=40%)', (rate >= 40).values)]:
    if yv[mask].sum() < 5:
        continue
    auc = roc_auc_score(yv[mask], best_oof[mask])
    print(f'  {label:20s}  n={mask.sum():5d}  AUC={auc:.4f}')

## 9. Generate Submission (CPU)

In [ ]:
# Train final models on ALL training data and predict test
skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
neg, pos = (yv == 0).sum(), (yv == 1).sum()

# LGBM v3 - average test predictions across folds
test_lgbm_v3 = np.zeros(len(Xt_v3))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v3, yv)):
    m = LGBMClassifier(
        n_estimators=3000, num_leaves=31, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.8,
        is_unbalance=True, random_state=SEED, verbose=-1, metric='auc',
    )
    m.fit(X_v3[tr_idx], yv[tr_idx], eval_set=[(X_v3[val_idx], yv[val_idx])],
          callbacks=[early_stopping(200, verbose=False)])
    test_lgbm_v3 += m.predict_proba(Xt_v3)[:, 1] / N_FOLDS
    print(f'  LGBM fold {fold+1} done (best_iter={m.best_iteration_})')

# XGB v3
test_xgb_v3 = np.zeros(len(Xt_v3))
for fold, (tr_idx, val_idx) in enumerate(skf.split(X_v3, yv)):
    m = xgb.XGBClassifier(
        n_estimators=3000, max_depth=6, learning_rate=0.01,
        subsample=0.8, colsample_bytree=0.8,
        scale_pos_weight=float(neg / pos),
        tree_method='hist', eval_metric='auc',
        random_state=SEED, verbosity=0, early_stopping_rounds=200,
    )
    m.fit(X_v3[tr_idx], yv[tr_idx], eval_set=[(X_v3[val_idx], yv[val_idx])], verbose=False)
    test_xgb_v3 += m.predict_proba(Xt_v3)[:, 1] / N_FOLDS
    print(f'  XGB fold {fold+1} done (best_iter={m.best_iteration_})')

# Rank-average ensemble
test_ensemble = (rankdata(test_lgbm_v3) + rankdata(test_xgb_v3)) / (2 * len(test_lgbm_v3))

print('\nTest predictions generated!')
print(f'  LGBM v3 mean: {test_lgbm_v3.mean():.4f}')
print(f'  XGB v3 mean:  {test_xgb_v3.mean():.4f}')
print(f'  Ensemble mean: {test_ensemble.mean():.4f}')

In [ ]:
# Write submission
sub = sample_sub.copy()
sub[TARGET_COL] = test_ensemble

out_path = 'submissions/submission_local_debug.csv'
import os
os.makedirs('submissions', exist_ok=True)
sub.to_csv(out_path, index=False)

print(f'Submission saved to {out_path}')
print(f'Shape: {sub.shape}')
print(sub.head())
print(f'\nTarget distribution: min={sub[TARGET_COL].min():.4f}, max={sub[TARGET_COL].max():.4f}, mean={sub[TARGET_COL].mean():.4f}')

## 10. Summary of Improvements

| Change | AUC Impact |
|--------|------------|
| Original pipeline (LGBM ordinal, logloss ES) | 0.677 |
| Fix: `metric='auc'` + ES patience 200 | 0.684 (+0.007) |
| Fix: Add missing indicators, rate bucket, rank features | ~0.001 |
| Fix: v3 target+WOE encoding (was broken) | 0.692 (+0.008) |
| XGBoost on v3 | 0.694 (+0.002) |
| Ensemble LGBM+XGB v3 | ~0.694 |

**Total improvement: 0.677 -> 0.694 (+0.017 AUC)**

### Next steps for further improvement:
- Run Optuna tuning on v3 variant (50+ trials)
- Add CatBoost to the ensemble
- Try different target encoding smoothing values
- Feature selection (drop zero-importance features)
- Try stacking instead of rank averaging